In [ ]:
# Cell 1: Environment & Dataset Assertions
import os
import sys
import time
import platform
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# Quietly pip install pytorchfire zarr pyarrow
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pytorchfire==1.0.3.post2", "zarr", "pyarrow"],
    capture_output=True,
    check=False,
)

# Detect Kaggle vs local execution
is_kaggle = Path("/kaggle").exists()
work_dir = Path("/kaggle/working") if is_kaggle else Path("output/kaggle_phase2d").resolve()
work_dir.mkdir(parents=True, exist_ok=True)
shards_dir = work_dir / "shards_v2d"
shards_dir.mkdir(parents=True, exist_ok=True)
reports_dir = work_dir / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

# Assert required Kaggle inputs exist and stop if missing
if is_kaggle:
    assert Path("/kaggle/input/wf052-code").exists(), (
        "Required dataset '/kaggle/input/wf052-code' is missing! "
        "Please attach the 'wf052-code' dataset to this Kaggle notebook."
    )
    assert Path("/kaggle/input/wf052-tiles-train").exists(), (
        "Required dataset '/kaggle/input/wf052-tiles-train' is missing! "
        "Please attach the 'wf052-tiles-train' dataset to this Kaggle notebook."
    )
    assert Path("/kaggle/input/wf052-teacher-config").exists(), (
        "Required dataset '/kaggle/input/wf052-teacher-config' is missing! "
        "Please attach the 'wf052-teacher-config' dataset to this Kaggle notebook."
    )

# Configure sys.path for wf052 code
code_candidates = [
    Path("/kaggle/input/wf052-code/src"),
    Path("/kaggle/input/wf052-code"),
    Path("src").resolve(),
    Path("../src").resolve(),
]
for c in code_candidates:
    if c.exists() and str(c) not in sys.path:
        sys.path.insert(0, str(c))

# Configure tiles path
tiles_candidates = [
    Path("/kaggle/input/wf052-tiles-train"),
    Path("data/tiles/fires").resolve(),
    Path("../data/tiles/fires").resolve(),
]
fires_dir = None
for t in tiles_candidates:
    if (t / "data" / "tiles" / "fires").exists():
        fires_dir = t / "data" / "tiles" / "fires"
        break
    elif t.exists() and (t / "caldor").exists():
        fires_dir = t
        break

# Configure teacher config path
cfg_candidates = [
    Path("/kaggle/input/wf052-teacher-config/teacher_v2_a6a92f81.yaml"),
    Path("out/wf052-teacher-config/teacher_v2_a6a92f81.yaml").resolve(),
    Path("configs/teacher_v2.yaml").resolve(),
]
teacher_cfg_path = None
for cp in cfg_candidates:
    if cp.exists():
        teacher_cfg_path = cp
        break

# Device and hardware detection
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else "cpu"
)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU (CPU)"
if device.type == "cuda":
    hw_name = f"NVIDIA {gpu_name}"
elif device.type == "mps":
    hw_name = "Apple Silicon (MPS)"
else:
    hw_name = f"CPU ({platform.processor() or platform.machine()})"


# Configure git commit info
git_commit_candidates = [
    Path("/kaggle/input/wf052-code/GIT_COMMIT.txt"),
    Path("GIT_COMMIT.txt").resolve(),
    Path("../GIT_COMMIT.txt").resolve(),
]
git_commit_info = "GIT_COMMIT.txt not found"
for gcp in git_commit_candidates:
    if gcp.exists():
        git_commit_info = gcp.read_text(encoding="utf-8").strip().replace('
', ' | ')
        break

print("=" * 60)
print("Cell 1: Environment & Hardware Configuration")
print("=" * 60)
print(f"Python Version:       {sys.version.split()[0]}")
print(f"PyTorch Version:      {torch.__version__}")
print(f"GPU Name:             {gpu_name}")
print(f"Active Device:        {device} ({hw_name})")
print(f"Working Dir:          {work_dir}")
print(f"Shards Dir:           {shards_dir}")
print(f"Fires Tiles Dir:      {fires_dir}")
print(f"Teacher Config Path:  {teacher_cfg_path}")
print(f"Git Commit:          {git_commit_info}")
print("=" * 60)


### 💾 Quick Save Checkpoint 1
> **Tip:** Environment verified. You may create a Quick Save version or proceed directly to Cell 2.


In [ ]:
# Cell 2: Teacher Timing Benchmark
import json
import yaml
from wf052.surrogate.standin_generator import (
    load_fire_rasters_crop,
    load_and_verify_frozen_teacher_v2,
    simulate_ensemble_teacher_v2,
)
from wf052.surrogate.dataset import calculate_upwind_third_ignition

print("=" * 60)
print("Cell 2: Teacher Timing Benchmark (Frozen teacher_v2 Ensemble)")
print("=" * 60)

# Load frozen teacher_v2 config and verify hashes
teacher_cfg = load_and_verify_frozen_teacher_v2(teacher_cfg_path)
print(f"Verified frozen teacher_v2 config: {teacher_cfg_path}")

# Load test tile (256x256 crop from Caldor training fire)
fire_ref = "caldor"
fbfm_base, elev_base, canopy_base, wind_base = load_fire_rasters_crop(
    fire_name=fire_ref,
    tile_size=(256, 256),
    fires_base_dir=str(fires_dir) if fires_dir else "data/tiles/fires",
)
ign_r, ign_c = calculate_upwind_third_ignition(wind_base, tile_shape=(256, 256))

# Warm-up run (1 run with K=4 seeds)
print("Running warm-up (1 run with K=4 seeds) ...", flush=True)
t_w0 = time.perf_counter()
_ = simulate_ensemble_teacher_v2(
    fbfm=fbfm_base,
    elevation_z=elev_base,
    canopy_cover=canopy_base,
    hourly_wind_uv=wind_base,
    ign_r=ign_r,
    ign_c=ign_c,
    teacher_config=teacher_cfg,
    k_ensemble=4,
    scenario_seed=42,
)
t_warmup = time.perf_counter() - t_w0
print(f"Warm-up complete: {t_warmup:.2f} s")

# Timed runs (3 runs with K=4 seeds)
timing_runs = []
for run_idx in range(3):
    t0 = time.perf_counter()
    _ = simulate_ensemble_teacher_v2(
        fbfm=fbfm_base,
        elevation_z=elev_base,
        canopy_cover=canopy_base,
        hourly_wind_uv=wind_base,
        ign_r=ign_r,
        ign_c=ign_c,
        teacher_config=teacher_cfg,
        k_ensemble=4,
        scenario_seed=100 + run_idx * 17,
    )
    dt = time.perf_counter() - t0
    timing_runs.append(dt)
    print(f"  Run {run_idx + 1}/3: {dt:.3f} s ({dt / 4.0:.3f} s/seed)")

sec_per_scenario = float(np.median(timing_runs))
sec_per_seed = sec_per_scenario / 4.0

print("-" * 60)
print(f"Measured Timing (256x256 crop, K=4 seeds, 12 h horizon):")
print(f"  Median per scenario (K=4):  {sec_per_scenario:.3f} s")
print(f"  Median per (tile, seed):    {sec_per_seed:.3f} s")
print(f"  Mean per scenario (K=4):    {np.mean(timing_runs):.3f} s")
print(f"  P95 per scenario (K=4):     {np.percentile(timing_runs, 95):.3f} s")
print("=" * 60)

with open(reports_dir / "teacher_timing_benchmark.json", "w", encoding="utf-8") as f:
    json.dump({
        "hardware": hw_name,
        "gpu_name": gpu_name,
        "fire_reference": fire_ref,
        "tile_size": [256, 256],
        "k_ensemble": 4,
        "warmup_seconds": t_warmup,
        "timed_runs_seconds": timing_runs,
        "sec_per_scenario": sec_per_scenario,
        "sec_per_seed": sec_per_seed,
    }, f, indent=2)


### 💾 Quick Save Checkpoint 2 (Resumable Shard Generation)
> **Tip:** Shards are saved incrementally to `/kaggle/working/shards_v2d/` (shards of up to 200 samples). If your Kaggle session limits approach, perform **Save Version -> Quick Save**. Re-running will automatically detect completed shards and resume without re-computation.


In [ ]:
# Cell 3: Resumable Shard Generation with Multi-Seed Ensemble
MODE = "smoke"  # "smoke" (20 scenarios, K=4) or "full" (from scenarios.yaml, K=16)

import os
import sys
import time
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
from wf052.surrogate.standin_generator import (
    generate_stratified_samples_rejection,
    save_shard_with_metadata,
)

print("=" * 60)
print(f"Cell 3: Multi-Seed Ensemble Shard Generation (MODE={MODE})")
print("=" * 60)

# Wall-clock cap for Cell 3 (os.environ.get("MAX_MINUTES", 40) in smoke)
max_minutes = float(os.environ.get("MAX_MINUTES", 40 if MODE == "smoke" else 480))
max_wall_sec = max_minutes * 60.0
cell3_t0 = time.perf_counter()

# Per-bin attempt cap
max_attempts_per_bin = int(os.environ.get("MAX_ATTEMPTS_PER_BIN", 300 if MODE == "smoke" else 1500))

# Scenarios configuration
scenarios_cfg_file = Path("configs/scenarios.yaml")
if not scenarios_cfg_file.exists() and Path("/kaggle/input/wf052-code/configs/scenarios.yaml").exists():
    scenarios_cfg_file = Path("/kaggle/input/wf052-code/configs/scenarios.yaml")

scenarios_cfg = {}
if scenarios_cfg_file.exists():
    with open(scenarios_cfg_file, "r", encoding="utf-8") as f:
        scenarios_cfg = yaml.safe_load(f)

strat_cfg = scenarios_cfg.get("stratified_sampling", {})
bin_edges = list(strat_cfg.get("bin_edges", [0.0, 0.002, 0.02, 0.1, 0.5, 1.0]))
bin_names = list(strat_cfg.get("bin_names", ["[0, 0.002)", "[0.002, 0.02)", "[0.02, 0.1)", "[0.1, 0.5)", "[0.5, 1.0]"]))
fires_list = ["bobcat", "caldor", "creek", "glass", "kincade", "tamarack", "zogg"]

if MODE == "smoke":
    k_ensemble = int(os.environ.get("K_ENSEMBLE", strat_cfg.get("smoke_k_ensemble", 4)))
    scenarios_per_fire = int(os.environ.get("SCENARIOS_PER_FIRE", strat_cfg.get("smoke_per_fire", 2)))
    max_minutes = float(os.environ.get("MAX_MINUTES", 20))
    max_attempts_per_bin = int(os.environ.get("MAX_ATTEMPTS_PER_BIN", 300))
    stall_cap = int(os.environ.get("STALL_CAP", 60))
    total_target_scenarios = len(fires_list) * scenarios_per_fire * len(bin_names)
else:
    k_ensemble = int(os.environ.get("K_ENSEMBLE", strat_cfg.get("full_k_ensemble", 16)))
    scenarios_per_fire = int(os.environ.get("SCENARIOS_PER_FIRE", strat_cfg.get("full_per_fire", 200)))
    max_minutes = float(os.environ.get("MAX_MINUTES", 480))
    max_attempts_per_bin = int(os.environ.get("MAX_ATTEMPTS_PER_BIN", 1500))
    stall_cap = int(os.environ.get("STALL_CAP", 400))
    total_target_scenarios = len(fires_list) * scenarios_per_fire

samples_per_shard = 200
num_shards = max(1, int(np.ceil(total_target_scenarios / samples_per_shard)))

# Effective sampler config print (Step 0b)
print(f"[Effective Sampler Config] Quotas per fire: {scenarios_per_fire} per bin ({scenarios_per_fire * len(bin_names)} total), K: {k_ensemble}, STALL_CAP: {stall_cap}, MAX_ATTEMPTS_PER_BIN: {max_attempts_per_bin}, MAX_MINUTES: {max_minutes}")

print(f"Configuration:")
print(f"  Mode:                     {MODE}")
print(f"  Scenarios per Fire:       {scenarios_per_fire}")
print(f"  Total Target Scenarios:   {total_target_scenarios}")
print(f"  Ensemble K seeds:         {k_ensemble}")
print(f"  Shards to produce:        {num_shards} (max {samples_per_shard} samples/shard)")
print(f"  Max Minutes Allowed:      {max_minutes:.1f} min ({max_wall_sec:.0f} s)")
print(f"  Max Attempts Per Bin:     {max_attempts_per_bin}")
print(f"  Stall Cap (STALL_CAP):    {stall_cap}")
print(f"  Output directory:         {shards_dir}")
print("-" * 60)

rejection_reports = []

def write_rejection_summary(reports, out_dir):
    if not reports:
        return
    summary_rows = []
    for b in range(4):
        summary_rows.append({
            "bin": b,
            "target": sum(r.target_quotas.get(b, 0) for r in reports),
            "generated": sum(r.generated_per_bin.get(b, 0) for r in reports),
            "rejected": sum(r.rejected_per_bin.get(b, 0) for r in reports),
            "unique": sum(r.unique_per_bin.get(b, 0) for r in reports),
        })
    df_sum = pd.DataFrame(summary_rows)
    df_sum.to_csv(out_dir / "rejection_summary.csv", index=False)
    print(f"Saved rejection summary to {out_dir / 'rejection_summary.csv'}")

capped = False

for shard_idx in range(num_shards):
    shard_id = f"shard_{shard_idx:03d}"
    shard_file = shards_dir / f"{shard_id}.npz"
    meta_file = shards_dir / f"{shard_id}_metadata.csv"

    if shard_file.exists() and meta_file.exists():
        print(f"[{shard_id}] Already exists ({shard_file.stat().st_size:,} bytes). Resumed/Skipped.")
        continue

    # Check wall-clock budget remaining
    elapsed_so_far = time.perf_counter() - cell3_t0
    sec_left = max_wall_sec - elapsed_so_far
    if sec_left <= 0:
        print(f"Cell 3 wall-clock cap reached ({elapsed_so_far/60:.1f} min elapsed >= {max_minutes:.1f} min). Exiting cleanly.")
        capped = True
        break

    print(f"[{shard_id}] Generating shard (K={k_ensemble}, time_left={sec_left/60:.1f}m) ...", flush=True)
    t_start = time.perf_counter()

    shard_samples, rep = generate_stratified_samples_rejection(
        fire_names=fires_list,
        mode=MODE,
        k_ensemble=k_ensemble,
        scenarios_per_fire=scenarios_per_fire,
        max_attempts=3000,
        max_attempts_per_bin=max_attempts_per_bin,
        stall_cap=stall_cap,
        max_wall_sec=sec_left,
        progress_interval=5,
        base_seed=42000 + shard_idx * 5000,
        tile_size=(256, 256),
        source_mode="teacher_v2_ensemble",
        teacher_config_path=teacher_cfg_path,
        fires_base_dir=str(fires_dir) if fires_dir else "data/tiles/fires",
        scenarios_config_path=scenarios_cfg_file if scenarios_cfg_file.exists() else "configs/scenarios.yaml",
    )
    rejection_reports.append(rep)

    if len(shard_samples) > 0:
        df_meta = save_shard_with_metadata(shard_samples, shard_id=shard_id, shards_dir=shards_dir)
        elapsed = time.perf_counter() - t_start
        print(f"[{shard_id}] Saved {len(shard_samples)} samples in {elapsed:.1f} s ({len(df_meta)} metadata rows).")
    else:
        print(f"[{shard_id}] 0 samples generated before cap was hit.")

    write_rejection_summary(rejection_reports, shards_dir)

    if (time.perf_counter() - cell3_t0) >= max_wall_sec:
        print(f"Cell 3 wall-clock limit of {max_minutes:.1f} minutes reached. Exiting cleanly.")
        capped = True
        break

write_rejection_summary(rejection_reports, shards_dir)
if capped:
    print("Execution capped cleanly; completed shards remain valid and resumable on next run.")


In [ ]:
# Cell 4: Diagnostics from Metadata CSV & Budget Estimation
import pandas as pd
import numpy as np

print("=" * 60)
print("Cell 4: Diagnostics from Shard Metadata CSVs")
print("=" * 60)

# Load all metadata CSVs
meta_files = sorted(shards_dir.glob("shard_*_metadata.csv"))
assert len(meta_files) > 0, f"No metadata CSV files found in {shards_dir}!"
df_meta = pd.concat([pd.read_csv(f) for f in meta_files], ignore_index=True)
print(f"Loaded {len(df_meta)} sample metadata rows from {len(meta_files)} shard CSV file(s).\n")

# 1. Per fire x bin table
print("--- 1. Per Fire x Bin Table (Target, Accepted, Attempts) ---")
bin_labels = ["[0, 0.002)", "[0.002, 0.02)", "[0.02, 0.1)", "[0.1, 0.5)", "[0.5, 1.0]"]
if "fire_id" in df_meta.columns and "bin_k1" in df_meta.columns:
    fires_present = sorted(df_meta["fire_id"].unique())
    header = ["Fire"] + [f"Bin {b} ({bin_labels[b]})" for b in range(5)] + ["Total Acc"]
    print("| " + " | ".join(header) + " |")
    print("| :--- | " + " | ".join([":---:"] * 5) + " | :---: |")
    for f in fires_present:
        sub_f = df_meta[df_meta["fire_id"] == f]
        row_strs = []
        for b in range(5):
            sub_b = sub_f[sub_f["bin_k1"] == b]
            acc_cnt = len(sub_b)
            att_cnt = int(sub_b["attempts_in_bin"].max()) if len(sub_b) > 0 and "attempts_in_bin" in sub_b.columns else acc_cnt
            row_strs.append(f"{acc_cnt} ({att_cnt} att)")
        print(f"| `{f}` | " + " | ".join(row_strs) + f" | {len(sub_f)} |")
else:
    bin_counts = df_meta["bin"].value_counts().to_dict()
    print("| Bin | Burned Frac Range | Unique Count |")
    print("| :--- | :---: | :---: |")
    for b in range(len(bin_labels)):
        print(f"| `{bin_labels[b]}` | bin {b} | {bin_counts.get(b, 0)} |")

# 2. Wind distribution table (mean, P10, P50, P90 across generated samples)
print("\n--- 2. Wind Distribution Across Samples ---")
pw = df_meta["peak_wind"]
print("| Metric | Peak Wind (m/s) |")
print("| :--- | :---: |")
print(f"| Mean | {pw.mean():.2f} |")
print(f"| P10  | {pw.quantile(0.10):.2f} |")
print(f"| P50 (Median) | {pw.quantile(0.50):.2f} |")
print(f"| P90  | {pw.quantile(0.90):.2f} |")
print(f"| Min  | {pw.min():.2f} |")
print(f"| Max  | {pw.max():.2f} |")

# 3. burn_prob histogram (0.0, 0-0.2, 0.2-0.5, 0.5-0.9, >0.9)
print("\n--- 3. burn_prob Histogram Across Samples ---")
bp = df_meta["burn_prob_mean"]
c_zero = int((bp == 0.0).sum())
c_0_02 = int(((bp > 0.0) & (bp <= 0.2)).sum())
c_02_05 = int(((bp > 0.2) & (bp <= 0.5)).sum())
c_05_09 = int(((bp > 0.5) & (bp <= 0.9)).sum())
c_gt_09 = int((bp > 0.9).sum())
n_tot = len(df_meta)

print("| Burn Probability Range | Sample Count | Fraction |")
print("| :--- | :---: | :---: |")
print(f"| `0.0` (Unburned)       | {c_zero} | {c_zero / n_tot * 100:.1f}% |")
print(f"| `(0.0, 0.2]`           | {c_0_02} | {c_0_02 / n_tot * 100:.1f}% |")
print(f"| `(0.2, 0.5]`           | {c_02_05} | {c_02_05 / n_tot * 100:.1f}% |")
print(f"| `(0.5, 0.9]`           | {c_05_09} | {c_05_09 / n_tot * 100:.1f}% |")
print(f"| `> 0.9`                | {c_gt_09} | {c_gt_09 / n_tot * 100:.1f}% |")

# 4. Censored fraction per quantile
print("\n--- 4. Censored Fraction per Quantile ---")
cens_q10 = 1.0 - df_meta["q10_burned_fraction"]
cens_q50 = df_meta["censored_fraction"]
cens_q90 = 1.0 - df_meta["q90_burned_fraction"]

print("| Quantile | Mean Censored Fraction | Median Censored Fraction |")
print("| :--- | :---: | :---: |")
print(f"| q10 (conservative / early) | {cens_q10.mean() * 100:.2f}% | {cens_q10.median() * 100:.2f}% |")
print(f"| q50 (median arrival)       | {cens_q50.mean() * 100:.2f}% | {cens_q50.median() * 100:.2f}% |")
print(f"| q90 (late arrival)         | {cens_q90.mean() * 100:.2f}% | {cens_q90.median() * 100:.2f}% |")

# 5. Count of scenarios where K runs differ
print("\n--- 5. Ensemble Diversity (Differing Runs) ---")
if "runs_differ" in df_meta.columns:
    diff_mask = df_meta["runs_differ"].astype(bool)
else:
    diff_mask = df_meta["q10_burned_fraction"] != df_meta["q90_burned_fraction"]
diff_count = int(diff_mask.sum())
print(f"Scenarios where K runs differ: {diff_count} / {n_tot} ({diff_count / n_tot * 100:.1f}%)")
print(f"Identical runs (zero spread):   {n_tot - diff_count} / {n_tot} ({(n_tot - diff_count) / n_tot * 100:.1f}%)")

# 6. Timing metrics & 28,800 s budget projection for K=8 and K=16
print("\n--- 6. Runtime & Capacity Projections (28,800 s budget) ---")
timing_file = reports_dir / "teacher_timing_benchmark.json"
sec_per_seed = 0.5
if timing_file.exists():
    with open(timing_file, "r", encoding="utf-8") as f:
        bench_data = json.load(f)
        sec_per_seed = float(bench_data.get("sec_per_seed", 0.5))
elif "measured_sec_per_seed" in locals():
    sec_per_seed = measured_sec_per_seed

tot_attempts = int(df_meta["attempts_in_bin"].sum()) if "attempts_in_bin" in df_meta.columns else n_tot
tot_accepted = n_tot
cand_per_acc = max(1.0, tot_attempts / max(1, tot_accepted))
sec_per_attempt = sec_per_seed
sec_per_accepted_k4 = cand_per_acc * sec_per_attempt + (4 - 1) * sec_per_seed
sec_per_accepted_k8 = cand_per_acc * sec_per_attempt + (8 - 1) * sec_per_seed
sec_per_accepted_k16 = cand_per_acc * sec_per_attempt + (16 - 1) * sec_per_seed

budget_sec = 28800.0  # 8 hours
proj_k8 = int(budget_sec // sec_per_accepted_k8)
proj_k16 = int(budget_sec // sec_per_accepted_k16)

print(f"Measured sec/attempt (K=1 prescreen): {sec_per_attempt:.3f} s")
print(f"Candidates per accepted sample:       {cand_per_acc:.2f}")
print(f"Sec/accepted sample (K=4 SMOKE):       {sec_per_accepted_k4:.2f} s")
print(f"Sec/accepted sample (K=8):             {sec_per_accepted_k8:.2f} s")
print(f"Sec/accepted sample (K=16):            {sec_per_accepted_k16:.2f} s")
print("| Configuration | Time per Accepted Sample | Projected in 28,800 s (8h) |")
print("| :---: | :---: | :---: |")
print(f"| **K = 8**  | {sec_per_accepted_k8:.2f} s | **{proj_k8:,} accepted samples** |")
print(f"| **K = 16** | {sec_per_accepted_k16:.2f} s | **{proj_k16:,} accepted samples** |")
print("=" * 60)


In [ ]:
# Cell 5: File Listing, Sizes, and SHA-256 Verification
import hashlib

print("=" * 60)
print("Cell 5: Shard Artifacts & Verification")
print("=" * 60)

# 1. Directory listing (ls -lh)
print("Directory listing for /kaggle/working/shards_v2d/:")
total_bytes = 0
all_files = sorted(shards_dir.glob("*"))
for fp in all_files:
    if fp.is_file():
        sz = fp.stat().st_size
        total_bytes += sz
        if sz < 1024:
            s_str = f"{sz} B"
        elif sz < 1024 * 1024:
            s_str = f"{sz / 1024:.2f} KB"
        else:
            s_str = f"{sz / (1024 * 1024):.2f} MB"
        print(f"  {fp.name:<35} {s_str:>10} ({sz:,} bytes)")

# Total size in MB
total_mb = total_bytes / (1024.0 * 1024.0)
print("-" * 60)
print(f"Total shards size: {total_mb:.2f} MB ({total_bytes:,} bytes)")

# SHA-256 of the first shard
first_shard_file = shards_dir / "shard_000.pkl"
if not first_shard_file.exists():
    pkl_files = sorted(shards_dir.glob("shard_*.pkl"))
    if pkl_files:
        first_shard_file = pkl_files[0]

if first_shard_file.exists():
    h = hashlib.sha256()
    with open(first_shard_file, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    first_sha256 = h.hexdigest()
    print(f"SHA-256 of first shard ({first_shard_file.name}):")
    print(f"  {first_sha256}")
else:
    print("No shard .pkl file found to hash!")
print("=" * 60)
print("Phase 2D Kaggle Execution Complete!")
